# EMA Cross + Bracket SL/TP — Custom vs. Nautilus-Internal 5-min Aggregation

Same EMA cross strategy run twice over **EUR/USD January-2024 1-min bars** (full month from
`Dataset/FX_yyyy/Fx/EUROUSD/2024/01/`):

| Run | Strategy | Subscribes to | Aggregation done by |
|-----|----------|---------------|---------------------|
| **internal** | `InternalEmaCrossStrategy` | `5-MINUTE-…-INTERNAL@1-MINUTE-EXTERNAL` | Nautilus DataEngine |
| **custom**   | `CustomAggEmaCrossStrategy` | `1-MINUTE-…-EXTERNAL` | `FiveMinAggregator` (this notebook) |

On every fast-EMA-vs-slow-EMA cross the strategy submits a **bracket** order: market entry
with SL and TP attached as OUO contingencies (when one fills, the other cancels).

Each run produces three CSV reports — fills, positions, account — written under
`reports_aggregator_demo/` with `internal_*` / `custom_*` prefixes (6 files total).

In [1]:
import sys
from dataclasses import dataclass, asdict, fields
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the project root robustly. Three search strategies, in order:
#   1. Walk UP from Path.cwd() looking for `core/instrument_factory.py`.
#   2. Walk UP from the notebook's own location (VS Code sets
#      `__vsc_ipynb_file__`; classic Jupyter sets `__session__`).
#   3. As a last resort, scan the immediate children of cwd.
def _find_project_root() -> Path:
    marker = Path("core") / "instrument_factory.py"

    def _walk_up(p: Path) -> Path | None:
        for parent in (p, *p.parents):
            if (parent / marker).is_file():
                return parent
        return None

    # 1. cwd
    found = _walk_up(Path.cwd())
    if found:
        return found
    # 2. notebook file location
    for var in ("__vsc_ipynb_file__", "__session__", "__file__"):
        nb = globals().get(var)
        if nb:
            found = _walk_up(Path(nb).resolve().parent)
            if found:
                return found
    # 3. children of cwd (VS Code may open an outer workspace folder)
    for child in Path.cwd().iterdir():
        if child.is_dir() and (child / marker).is_file():
            return child
    raise RuntimeError(
        f"could not find project root (no `{marker}`) from cwd={Path.cwd()}"
    )


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = Path(r"C:/Users/HP/Desktop/MS/Dataset/FX_yyyy/Fx/EUROUSD/2024/01")
REPORTS_DIR = PROJECT_ROOT / "reports_aggregator_demo"
REPORTS_DIR.mkdir(exist_ok=True)
print(f"cwd         : {Path.cwd()}")
print(f"project root: {PROJECT_ROOT}")
print(f"reports dir : {REPORTS_DIR}")

BASE, QUOTE, VENUE_NAME, PRICE_TYPE = "EUR", "USD", "FOREX_MS", "ASK"
OUTPUT_TZ = "Asia/Kolkata"

# Strategy / risk parameters
FAST_EMA = 9
SLOW_EMA = 21
TRADE_SIZE = Decimal("100000")  # 1 standard FX lot
PIP = 0.0001                    # EUR/USD pip
SL_PIPS = 20                    # stop-loss distance from entry
TP_PIPS = 40                    # take-profit distance (1:2 R:R)
STARTING_BALANCE_USD = 1_000_000

cwd         : d:\mcube_test_version_1_updated\m-cube_version1\ipynb
project root: d:\mcube_test_version_1_updated\m-cube_version1
reports dir : d:\mcube_test_version_1_updated\m-cube_version1\reports_aggregator_demo


In [2]:
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import (
    BacktestEngineConfig, LoggingConfig, PositiveInt, RiskEngineConfig, StrategyConfig,
)
from nautilus_trader.indicators import ExponentialMovingAverage
from nautilus_trader.model import TraderId
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AccountType, OmsType, OrderSide, TimeInForce
from nautilus_trader.model.identifiers import InstrumentId, Venue
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.wranglers import BarDataWrangler
from nautilus_trader.trading.strategy import Strategy

from core.instrument_factory import create_instrument

## Load the full month — both ASK and BID sides

Each day-folder ships an `*_ASK_OHLCV.csv` and an `*_BID_OHLCV.csv`. We load **both**
sides into separate UTC-indexed DataFrames.

> **Why both sides?** The strategies only need to read the ASK feed for their EMAs,
> but the simulated FX matching engine needs *both* BID and ASK to construct quotes
> for fill simulation. Loading only one side causes every market entry to be rejected
> with `'no market for EURUSD.FOREX_MS'` and the SL/TP children cascade-reject. The
> project already encodes this rule in [`core/backtest_runner.py`](core/backtest_runner.py)
> via `_pair_bid_ask_bar_type` — we apply the same convention here.

In [3]:
def load_month(data_dir: Path, side: str) -> pd.DataFrame:
    """Concatenate every `*_<side>_OHLCV.csv` under `data_dir/<DD>/` into one UTC frame."""
    frames: list[pd.DataFrame] = []
    for day_dir in sorted(data_dir.iterdir()):
        if not day_dir.is_dir():
            continue
        for csv_path in sorted(day_dir.glob(f"*_{side}_OHLCV.csv")):
            frames.append(pd.read_csv(csv_path))
    if not frames:
        raise RuntimeError(f"No CSVs found for side={side} in {data_dir}")
    df = pd.concat(frames, ignore_index=True)
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%d.%m.%Y %H:%M:%S.%f GMT%z", utc=True
    )
    df = df.set_index("timestamp").sort_index()
    df = df[~df.index.duplicated(keep="last")]
    return df[["open", "high", "low", "close", "volume"]].astype(float)


df_1min_ask = load_month(DATA_DIR, side="ASK")
df_1min_bid = load_month(DATA_DIR, side="BID")
print(f"loaded ASK={len(df_1min_ask):,}  BID={len(df_1min_bid):,} 1-min bars")
print(f"range (UTC): {df_1min_ask.index.min()} → {df_1min_ask.index.max()}")
print(f"range (IST): {df_1min_ask.index.min().tz_convert(OUTPUT_TZ)} → {df_1min_ask.index.max().tz_convert(OUTPUT_TZ)}")
df_1min_ask.head()

loaded ASK=31,470  BID=31,470 1-min bars
range (UTC): 2024-01-01 22:00:00+00:00 → 2024-01-31 18:29:00+00:00
range (IST): 2024-01-02 03:30:00+05:30 → 2024-01-31 23:59:00+05:30


,open,high,low,close,volume
timestamp,,,,,
2024-01-01 22:00:00+00:00,1.10481,1.10481,1.10476,1.10480,5.9
2024-01-01 22:01:00+00:00,1.10477,1.10479,1.10477,1.10479,26.1
2024-01-01 22:02:00+00:00,1.10478,1.10479,1.10477,1.10479,27.9
2024-01-01 22:03:00+00:00,1.10478,1.10479,1.10478,1.10479,9.0
2024-01-01 22:04:00+00:00,1.10478,1.10480,1.10478,1.10480,11.7


## Wrangle into Nautilus `Bar` objects (1-min EXTERNAL, ASK + BID)

Both sides are wrangled into Nautilus `Bar` lists and concatenated as `ALL_BARS`. This
combined list is what the engine sees via `engine.add_data(ALL_BARS)`. The strategies
will *subscribe* only to the ASK side (or to the 5-min INTERNAL composite that draws
from the ASK 1-min stream); the BID stream is purely there so the matching engine has
a complete L1 quote to fill against.

In [4]:
instrument = create_instrument(BASE, QUOTE, venue=VENUE_NAME)

BAR_TYPE_1MIN_ASK = BarType.from_str(f"{instrument.id}-1-MINUTE-ASK-EXTERNAL")
BAR_TYPE_1MIN_BID = BarType.from_str(f"{instrument.id}-1-MINUTE-BID-EXTERNAL")
BAR_TYPE_5MIN_INTERNAL = BarType.from_str(
    f"{instrument.id}-5-MINUTE-ASK-INTERNAL@1-MINUTE-EXTERNAL"
)

BARS_1MIN_ASK = BarDataWrangler(BAR_TYPE_1MIN_ASK, instrument).process(df_1min_ask)
BARS_1MIN_BID = BarDataWrangler(BAR_TYPE_1MIN_BID, instrument).process(df_1min_bid)
ALL_BARS = BARS_1MIN_ASK + BARS_1MIN_BID

print(f"wrangled ASK={len(BARS_1MIN_ASK):,}  BID={len(BARS_1MIN_BID):,}  total={len(ALL_BARS):,}")
print(f"strategy bar type (1-min ASK): {BAR_TYPE_1MIN_ASK}")
print(f"composite bar type (5-min INTERNAL): {BAR_TYPE_5MIN_INTERNAL}")

wrangled ASK=31,470  BID=31,470  total=62,940
strategy bar type (1-min ASK): EURUSD.FOREX_MS-1-MINUTE-ASK-EXTERNAL
composite bar type (5-min INTERNAL): EURUSD.FOREX_MS-5-MINUTE-ASK-INTERNAL@1-MINUTE-EXTERNAL


## `Bar5MinFeatures` + `FiveMinAggregator`

Re-used verbatim from `aggregate_1min_to_5min.ipynb` — this is the same custom
aggregator class as before. The strategy in run-2 only consumes `b5.close` from the
emitted bars to feed its EMAs, but the aggregator still computes the full feature set
so you can inspect `b5.atr_14`, `b5.rsi_14`, etc. inside `on_bar` if you want to add
feature-based filters.

In [5]:
@dataclass
class Bar5MinFeatures:
    timestamp: pd.Timestamp
    open: float
    high: float
    low: float
    close: float
    volume: float
    n_ticks: int
    range: float
    body: float
    upper_wick: float
    lower_wick: float
    log_return: float | None
    pct_return: float | None
    typical_price: float
    vwap: float
    tr: float
    atr_14: float
    ema_9: float
    ema_21: float
    rsi_14: float | None

    @classmethod
    def field_names(cls) -> list[str]:
        return [f.name for f in fields(cls)]

    def to_row(self) -> dict:
        return asdict(self)


class FiveMinAggregator:
    """Buffer 1-min bars, emit a `Bar5MinFeatures` at every N-min boundary."""

    def __init__(
        self,
        target_minutes: int = 5,
        ema_short_period: int = 9,
        ema_long_period: int = 21,
        rsi_period: int = 14,
        atr_period: int = 14,
    ) -> None:
        self.target_minutes = target_minutes
        self._alpha_ema_s = 2.0 / (ema_short_period + 1)
        self._alpha_ema_l = 2.0 / (ema_long_period + 1)
        self._alpha_rsi = 1.0 / rsi_period
        self._alpha_atr = 1.0 / atr_period
        self._buffer: list[Bar] = []
        self._window_close: pd.Timestamp | None = None
        self._prev_close: float | None = None
        self._ema_s: float | None = None
        self._ema_l: float | None = None
        self._avg_gain: float = 0.0
        self._avg_loss: float = 0.0
        self._atr: float | None = None
        self._cum_pv: float = 0.0
        self._cum_v: float = 0.0

    @staticmethod
    def _bar_ts(bar: Bar) -> pd.Timestamp:
        return pd.Timestamp(bar.ts_event, unit="ns", tz="UTC")

    def on_bar(self, bar: Bar) -> Bar5MinFeatures | None:
        ts = self._bar_ts(bar)
        bucket_close = ts.ceil(f"{self.target_minutes}min")
        emitted: Bar5MinFeatures | None = None
        if self._window_close is None:
            self._window_close = bucket_close
        elif bucket_close != self._window_close:
            emitted = self._close_window()
            self._window_close = bucket_close
        self._buffer.append(bar)
        return emitted

    def flush(self) -> Bar5MinFeatures | None:
        if self._buffer:
            return self._close_window()
        return None

    def _close_window(self) -> Bar5MinFeatures:
        bars = self._buffer
        opens = float(bars[0].open)
        highs = max(float(b.high) for b in bars)
        lows = min(float(b.low) for b in bars)
        closes = float(bars[-1].close)
        volume = sum(float(b.volume) for b in bars)
        n_ticks = len(bars)
        rng = highs - lows
        body = abs(closes - opens)
        upper_wick = highs - max(opens, closes)
        lower_wick = min(opens, closes) - lows
        if self._prev_close is None or self._prev_close == 0:
            log_ret, pct_ret = None, None
        else:
            log_ret = float(np.log(closes / self._prev_close))
            pct_ret = closes / self._prev_close - 1.0
        typical = (highs + lows + closes) / 3.0
        self._cum_pv += typical * volume
        self._cum_v += volume
        vwap = self._cum_pv / self._cum_v if self._cum_v > 0 else typical
        tr = rng if self._prev_close is None else max(
            rng, abs(highs - self._prev_close), abs(lows - self._prev_close)
        )
        self._atr = tr if self._atr is None else self._atr + self._alpha_atr * (tr - self._atr)
        self._ema_s = closes if self._ema_s is None else self._ema_s + self._alpha_ema_s * (closes - self._ema_s)
        self._ema_l = closes if self._ema_l is None else self._ema_l + self._alpha_ema_l * (closes - self._ema_l)
        rsi = None
        if self._prev_close is not None:
            change = closes - self._prev_close
            gain, loss = max(change, 0.0), max(-change, 0.0)
            self._avg_gain += self._alpha_rsi * (gain - self._avg_gain)
            self._avg_loss += self._alpha_rsi * (loss - self._avg_loss)
            if self._avg_loss > 0:
                rs = self._avg_gain / self._avg_loss
                rsi = 100.0 - 100.0 / (1.0 + rs)
            elif self._avg_gain > 0:
                rsi = 100.0
        out = Bar5MinFeatures(
            timestamp=self._window_close,
            open=opens, high=highs, low=lows, close=closes, volume=volume,
            n_ticks=n_ticks, range=rng, body=body,
            upper_wick=upper_wick, lower_wick=lower_wick,
            log_return=log_ret, pct_return=pct_ret,
            typical_price=typical, vwap=vwap,
            tr=tr, atr_14=self._atr,
            ema_9=self._ema_s, ema_21=self._ema_l, rsi_14=rsi,
        )
        self._prev_close = closes
        self._buffer = []
        return out

## Shared bracket-order helper

Both strategies wrap the same logic for submitting an EMA-cross entry: `order_factory.bracket()`
creates one entry order plus an SL (`STOP_MARKET`) and TP (`LIMIT`) child, linked under
an OUO contingency so when one child fills the other auto-cancels. SL and TP are placed
`SL_PIPS` / `TP_PIPS` away from the bar's close (the entry-price estimate).

In [6]:
def submit_bracket(strategy: Strategy, instrument, side: OrderSide, entry_price: float) -> None:
    if side == OrderSide.BUY:
        sl = instrument.make_price(entry_price - SL_PIPS * PIP)
        tp = instrument.make_price(entry_price + TP_PIPS * PIP)
    else:
        sl = instrument.make_price(entry_price + SL_PIPS * PIP)
        tp = instrument.make_price(entry_price - TP_PIPS * PIP)
    order_list = strategy.order_factory.bracket(
        instrument_id=instrument.id,
        order_side=side,
        quantity=instrument.make_qty(TRADE_SIZE),
        sl_trigger_price=sl,
        tp_price=tp,
        time_in_force=TimeInForce.GTC,
    )
    strategy.submit_order_list(order_list)

## Strategy 1 — `InternalEmaCrossStrategy`

Subscribes to the 5-min INTERNAL composite bar type. EMAs are registered against the
5-min bar via `register_indicator_for_bars`, so Nautilus updates them automatically as
5-min bars get aggregated and delivered to `on_bar`. On a fast-vs-slow cross, if flat,
submit a bracket order; otherwise wait for the existing bracket's SL or TP to close it.

In [7]:
class InternalEmaCrossConfig(StrategyConfig, frozen=True):
    instrument_id: InstrumentId
    bar_type: BarType  # 5-MIN INTERNAL@1-MIN EXTERNAL composite
    fast_ema_period: PositiveInt = FAST_EMA
    slow_ema_period: PositiveInt = SLOW_EMA


class InternalEmaCrossStrategy(Strategy):
    """EMA cross + bracket SL/TP. Aggregation done by Nautilus DataEngine."""

    def __init__(self, config: InternalEmaCrossConfig) -> None:
        super().__init__(config)
        self.instrument = None
        self.fast_ema = ExponentialMovingAverage(config.fast_ema_period)
        self.slow_ema = ExponentialMovingAverage(config.slow_ema_period)
        self._prev_fast_above: bool | None = None
        self._n_brackets = 0

    def on_start(self) -> None:
        self.instrument = self.cache.instrument(self.config.instrument_id)
        self.register_indicator_for_bars(self.config.bar_type, self.fast_ema)
        self.register_indicator_for_bars(self.config.bar_type, self.slow_ema)
        self.subscribe_bars(self.config.bar_type)

    def on_bar(self, bar: Bar) -> None:
        if not self.indicators_initialized():
            return
        fast_above = self.fast_ema.value > self.slow_ema.value
        if self._prev_fast_above is None:
            self._prev_fast_above = fast_above
            return
        crossed_up = fast_above and not self._prev_fast_above
        crossed_down = (not fast_above) and self._prev_fast_above
        self._prev_fast_above = fast_above
        if not (crossed_up or crossed_down):
            return
        if not self.portfolio.is_flat(self.config.instrument_id):
            return  # bracket from previous cross is still working — let it run to SL/TP
        side = OrderSide.BUY if crossed_up else OrderSide.SELL
        submit_bracket(self, self.instrument, side, float(bar.close))
        self._n_brackets += 1

    def on_stop(self) -> None:
        self.cancel_all_orders(self.config.instrument_id)
        self.close_all_positions(self.config.instrument_id)
        self.log.info(f"submitted {self._n_brackets} brackets")

## Strategy 2 — `CustomAggEmaCrossStrategy`

Subscribes to 1-min EXTERNAL bars, pushes them through `FiveMinAggregator`, and runs
EMA logic only when the aggregator emits a 5-min bar. Indicators are updated manually
via `update_raw(close)` since they aren't bound to a bar type the engine knows about.

In [8]:
class CustomAggEmaCrossConfig(StrategyConfig, frozen=True):
    instrument_id: InstrumentId
    bar_type: BarType  # 1-MIN EXTERNAL
    fast_ema_period: PositiveInt = FAST_EMA
    slow_ema_period: PositiveInt = SLOW_EMA
    target_minutes: int = 5


class CustomAggEmaCrossStrategy(Strategy):
    """EMA cross + bracket SL/TP. Aggregation done by FiveMinAggregator."""

    def __init__(self, config: CustomAggEmaCrossConfig) -> None:
        super().__init__(config)
        self.instrument = None
        self._aggregator = FiveMinAggregator(target_minutes=config.target_minutes)
        self.fast_ema = ExponentialMovingAverage(config.fast_ema_period)
        self.slow_ema = ExponentialMovingAverage(config.slow_ema_period)
        self._prev_fast_above: bool | None = None
        self._n_brackets = 0

    def on_start(self) -> None:
        self.instrument = self.cache.instrument(self.config.instrument_id)
        self.subscribe_bars(self.config.bar_type)  # 1-min EXTERNAL

    def on_bar(self, bar: Bar) -> None:
        b5 = self._aggregator.on_bar(bar)
        if b5 is None:
            return  # still inside the current 5-min window
        # 5-min bar just closed → drive EMAs from it
        self.fast_ema.update_raw(b5.close)
        self.slow_ema.update_raw(b5.close)
        if not (self.fast_ema.initialized and self.slow_ema.initialized):
            return
        fast_above = self.fast_ema.value > self.slow_ema.value
        if self._prev_fast_above is None:
            self._prev_fast_above = fast_above
            return
        crossed_up = fast_above and not self._prev_fast_above
        crossed_down = (not fast_above) and self._prev_fast_above
        self._prev_fast_above = fast_above
        if not (crossed_up or crossed_down):
            return
        if not self.portfolio.is_flat(self.config.instrument_id):
            return
        side = OrderSide.BUY if crossed_up else OrderSide.SELL
        submit_bracket(self, self.instrument, side, b5.close)
        self._n_brackets += 1

    def on_stop(self) -> None:
        self._aggregator.flush()  # drain trailing partial window (no order action)
        self.cancel_all_orders(self.config.instrument_id)
        self.close_all_positions(self.config.instrument_id)
        self.log.info(f"submitted {self._n_brackets} brackets")

## Engine harness + report writer

`run_backtest(strategy, prefix)`:
1. Builds a fresh `BacktestEngine` with a margin USD account
2. Loads the FX venue, instrument, and the 1-min EXTERNAL bar stream
3. Attaches the strategy and runs
4. Pulls the three built-in reports (`generate_order_fills_report`,
   `generate_positions_report`, `generate_account_report`) and writes them to
   `reports_aggregator_demo/{prefix}_*.csv`

In [9]:
def _save_reports(engine: BacktestEngine, prefix: str, venue: Venue) -> dict[str, int]:
    counts: dict[str, int] = {}

    fills = engine.trader.generate_order_fills_report()
    fills.to_csv(REPORTS_DIR / f"{prefix}_order_fills.csv")
    counts["fills"] = len(fills)

    positions = engine.trader.generate_positions_report()
    positions.to_csv(REPORTS_DIR / f"{prefix}_positions.csv")
    counts["positions"] = len(positions)

    account = engine.trader.generate_account_report(venue)
    account.to_csv(REPORTS_DIR / f"{prefix}_account.csv")
    counts["account"] = len(account)

    print(f"[{prefix}] reports → {REPORTS_DIR}")
    for k, v in counts.items():
        print(f"    {prefix}_{k}.csv  ({v} rows)")
    return counts


def run_backtest(strategy: Strategy, prefix: str) -> dict[str, int]:
    engine = BacktestEngine(
        config=BacktestEngineConfig(
            trader_id=TraderId("AGG-DEMO-001"),
            logging=LoggingConfig(bypass_logging=True),
            risk_engine=RiskEngineConfig(bypass=True),
            run_analysis=False,
        )
    )
    venue = instrument.id.venue
    engine.add_venue(
        venue=venue,
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        starting_balances=[Money(STARTING_BALANCE_USD, USD)],
        base_currency=USD,
        default_leverage=Decimal(30),  # typical FX leverage
    )
    engine.add_instrument(instrument)
    engine.add_data(ALL_BARS)  # BID + ASK; engine sorts by ts internally
    engine.add_strategy(strategy)
    engine.run()
    counts = _save_reports(engine, prefix, venue)
    engine.dispose()
    return counts

### Run 1 — internal aggregation (Nautilus DataEngine builds the 5-min bars)

In [10]:
internal_strategy = InternalEmaCrossStrategy(
    InternalEmaCrossConfig(instrument_id=instrument.id, bar_type=BAR_TYPE_5MIN_INTERNAL)
)
counts_internal = run_backtest(internal_strategy, prefix="internal")

[internal] reports → d:\mcube_test_version_1_updated\m-cube_version1\reports_aggregator_demo
    internal_fills.csv  (78 rows)
    internal_positions.csv  (39 rows)
    internal_account.csv  (315 rows)


### Run 2 — custom aggregation (`FiveMinAggregator` builds the 5-min bars)

In [11]:
custom_strategy = CustomAggEmaCrossStrategy(
    CustomAggEmaCrossConfig(instrument_id=instrument.id, bar_type=BAR_TYPE_1MIN_ASK)
)
counts_custom = run_backtest(custom_strategy, prefix="custom")

[custom] reports → d:\mcube_test_version_1_updated\m-cube_version1\reports_aggregator_demo
    custom_fills.csv  (80 rows)
    custom_positions.csv  (40 rows)
    custom_account.csv  (323 rows)


## Compare results

In [12]:
summary = pd.DataFrame({"internal": counts_internal, "custom": counts_custom})
print(summary)
print()
print("files written:")
for p in sorted(REPORTS_DIR.glob("*.csv")):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

           internal  custom
fills            78      80
positions        39      40
account         315     323

files written:
  custom_account.csv  (60,676 bytes)
  custom_order_fills.csv  (40,337 bytes)
  custom_positions.csv  (16,478 bytes)
  internal_account.csv  (59,178 bytes)
  internal_order_fills.csv  (39,133 bytes)
  internal_positions.csv  (15,981 bytes)


In [13]:
# Peek at a few rows from each report
for prefix in ("internal", "custom"):
    print(f"\n=== {prefix}_order_fills.csv (head) ===")
    print(pd.read_csv(REPORTS_DIR / f"{prefix}_order_fills.csv").head())
    print(f"\n=== {prefix}_positions.csv (head) ===")
    print(pd.read_csv(REPORTS_DIR / f"{prefix}_positions.csv").head())
    print(f"\n=== {prefix}_account.csv (head) ===")
    print(pd.read_csv(REPORTS_DIR / f"{prefix}_account.csv").head())


=== internal_order_fills.csv (head) ===
               client_order_id     trader_id                   strategy_id  \
0  O-20240102-011500-001-000-1  AGG-DEMO-001  InternalEmaCrossStrategy-000   
1  O-20240102-011500-001-000-2  AGG-DEMO-001  InternalEmaCrossStrategy-000   
2  O-20240102-052000-001-000-4  AGG-DEMO-001  InternalEmaCrossStrategy-000   
3  O-20240102-052000-001-000-5  AGG-DEMO-001  InternalEmaCrossStrategy-000   
4  O-20240102-150000-001-000-7  AGG-DEMO-001  InternalEmaCrossStrategy-000   

     instrument_id  venue_order_id  \
0  EURUSD.FOREX_MS  FOREX_MS-1-001   
1  EURUSD.FOREX_MS  FOREX_MS-1-002   
2  EURUSD.FOREX_MS  FOREX_MS-1-004   
3  EURUSD.FOREX_MS  FOREX_MS-1-005   
4  EURUSD.FOREX_MS  FOREX_MS-1-007   

                                    position_id    account_id   last_trade_id  \
0  EURUSD.FOREX_MS-InternalEmaCrossStrategy-000  FOREX_MS-001  FOREX_MS-1-002   
1  EURUSD.FOREX_MS-InternalEmaCrossStrategy-000  FOREX_MS-001  FOREX_MS-1-004   
2  EURUSD.FOREX_MS